# Phase 4: Integration 2: Similarity Triangle Edge Measurement

## Overview
Formally define, measure, and visualize the three edges of the S-F-R Similarity Triangle.
This is a generic framework: for any set of circuits discovered under different conditions,
compute how strongly structural similarity correlates with functional similarity,
how strongly representational similarity correlates with functional similarity,
and whether structural and representational similarity are related.

## The Three Edges
- **S-F**: Structural metrics vs Functional metrics
- **S-R**: Structural metrics vs Representational metrics
- **R-F**: Representational metrics vs Functional metrics

## Key Analyses
1. Overall vs within-model Spearman correlations for each edge
2. Simpson's paradox detection (sign reversal or magnitude collapse)
3. Partial correlations controlling for model
4. Within-model predictive power (LOO-CV R²)
5. Incremental prediction: does one perspective add information beyond another?

## Output Files
- `triangle_sf_edge.csv`, `triangle_sr_edge.csv`, `triangle_rf_edge.csv`
- `incremental_prediction.csv`
- Visualizations: T4_01 through T4_04

In [1]:
import sys
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
PROJECT_ROOT = Path("LSC_circuit_analysis")
PHASE4_DIR = PROJECT_ROOT / "04_Phase_Integration"
ANALYSIS_DIR = PHASE4_DIR / "outputs" / "analysis"
VIZ_DIR = PHASE4_DIR / "outputs" / "viz"

# Load unified tables from NB01
df_pc = pd.read_csv(ANALYSIS_DIR / "unified_per_circuit.csv")
df_pw = pd.read_csv(ANALYSIS_DIR / "unified_pairwise.csv")
df_ml = pd.read_csv(ANALYSIS_DIR / "unified_model_level.csv")
df_cat = pd.read_csv(ANALYSIS_DIR / "metric_catalog.csv")

print(f"Per-circuit: {df_pc.shape}")
print(f"Pairwise: {df_pw.shape}")
print(f"Model-level: {df_ml.shape}")
print(f"Metric catalog: {df_cat.shape}")

# Define metric groups by perspective from catalog
STRUCTURAL_METRICS = df_cat[df_cat["perspective"] == "structural"]["metric"].tolist()
FUNCTIONAL_METRICS = df_cat[df_cat["perspective"] == "functional"]["metric"].tolist()
REPRESENTATIONAL_METRICS = df_cat[df_cat["perspective"] == "representational"][
    "metric"
].tolist()


# Filter to metrics actually in per-circuit table and with variance
def get_variable_metrics(df, metric_list):
    """Return metrics present in df with non-zero variance."""
    return [m for m in metric_list if m in df.columns and df[m].std() > 1e-10]


S_METRICS = get_variable_metrics(df_pc, STRUCTURAL_METRICS)
F_METRICS = get_variable_metrics(df_pc, FUNCTIONAL_METRICS)
R_METRICS = get_variable_metrics(df_pc, REPRESENTATIONAL_METRICS)

MODELS = sorted(df_pc["model"].unique())

print(f"\nStructural metrics ({len(S_METRICS)}): {S_METRICS}")
print(f"Functional metrics ({len(F_METRICS)}): {F_METRICS}")
print(f"Representational metrics ({len(R_METRICS)}): {R_METRICS}")
print(f"Models: {MODELS}")

# Style
MODEL_COLORS = {
    "pythia-70m": "#1f77b4",
    "pythia-160m": "#ff7f0e",
    "pythia-410m": "#2ca02c",
    "pythia-1b": "#d62728",
    "pythia-1.4b": "#9467bd",
}

Per-circuit: (60, 32)
Pairwise: (50, 13)
Model-level: (5, 74)
Metric catalog: (27, 6)

Structural metrics (10): ['edge_fraction', 'skip_fraction', 'attn_fraction', 'mlp_fraction', 'resid_fraction', 'head_participation_rate', 'active_heads', 'mean_edges_per_head', 'n_universal', 'universal_fraction']
Functional metrics (8): ['n_edges', 'total_edges', 'size_fraction', 'base_accuracy', 'circuit_accuracy', 'circuit_kl_div', 'retention_ratio', 'completeness']
Representational metrics (8): ['convergence_layer', 'final_prob_correct', 'peak_probe_accuracy', 'peak_probe_layer', 'peak_separation_ratio', 'peak_mi_probe', 'peak_mi_probe_layer', 'peak_efficiency']
Models: ['pythia-1.4b', 'pythia-160m', 'pythia-1b', 'pythia-410m', 'pythia-70m']


## 1. S-F Edge: Structure vs Function

Compute Spearman correlations between each structural metric and each functional metric,
both overall (all circuits pooled) and within each model. Check for Simpson's paradox.

In [2]:
def compute_edge_correlations(
    df, metrics_a, metrics_b, label_a="metric_a", label_b="metric_b"
):
    """Compute overall and within-model Spearman correlations for all metric pairs.

    Returns DataFrame with columns:
        metric_a, metric_b, rho_overall, p_overall,
        rho_within_mean, rho_within_std, n_within_significant,
        n_models, simpsons_flag, partial_rho, partial_p
    """
    rows = []
    models = sorted(df["model"].unique())

    for ma, mb in product(metrics_a, metrics_b):
        if ma == mb:
            continue

        x_all = df[ma].values
        y_all = df[mb].values

        # Skip if no variance
        if np.std(x_all) < 1e-10 or np.std(y_all) < 1e-10:
            continue

        # Overall correlation
        rho_overall, p_overall = stats.spearmanr(x_all, y_all)

        # Within-model correlations
        within_rhos = []
        within_ps = []
        for model in models:
            mask = df["model"] == model
            x_m = df.loc[mask, ma].values
            y_m = df.loc[mask, mb].values
            if np.std(x_m) < 1e-10 or np.std(y_m) < 1e-10:
                continue
            r, p = stats.spearmanr(x_m, y_m)
            if not np.isnan(r):
                within_rhos.append(r)
                within_ps.append(p)

        rho_within_mean = np.mean(within_rhos) if within_rhos else np.nan
        rho_within_std = np.std(within_rhos) if within_rhos else np.nan
        n_within_sig = sum(1 for p in within_ps if p < 0.05)
        n_models_valid = len(within_rhos)

        # Simpson's paradox: sign reversal or >50% magnitude drop
        simpsons_flag = False
        if not np.isnan(rho_within_mean) and abs(rho_overall) > 0.1:
            if np.sign(rho_overall) != np.sign(rho_within_mean):
                simpsons_flag = True  # Sign reversal
            elif abs(rho_within_mean) < 0.5 * abs(rho_overall):
                simpsons_flag = True  # >50% magnitude drop

        # Partial correlation controlling for model (rank-based)
        # Encode model as numeric, compute partial Spearman
        model_code = pd.Categorical(df["model"]).codes
        try:
            # Partial correlation: correlate residuals after regressing out model
            from sklearn.linear_model import LinearRegression

            lr = LinearRegression()
            mc = model_code.reshape(-1, 1)
            x_resid = x_all - lr.fit(mc, x_all).predict(mc)
            y_resid = y_all - lr.fit(mc, y_all).predict(mc)
            partial_rho, partial_p = stats.spearmanr(x_resid, y_resid)
        except Exception:
            partial_rho, partial_p = np.nan, np.nan

        rows.append(
            {
                label_a: ma,
                label_b: mb,
                "rho_overall": rho_overall,
                "p_overall": p_overall,
                "rho_within_mean": rho_within_mean,
                "rho_within_std": rho_within_std,
                "n_within_significant": n_within_sig,
                "n_models": n_models_valid,
                "simpsons_flag": simpsons_flag,
                "partial_rho": partial_rho,
                "partial_p": partial_p,
            }
        )

    return pd.DataFrame(rows)


# Compute S-F edge
df_sf = compute_edge_correlations(
    df_pc, S_METRICS, F_METRICS, label_a="structural", label_b="functional"
)
df_sf = df_sf.sort_values("p_overall").reset_index(drop=True)
df_sf.to_csv(ANALYSIS_DIR / "triangle_sf_edge.csv", index=False)

print(f"S-F Edge: {len(df_sf)} metric pairs")
print(f"  Significant overall (p<0.05): {(df_sf['p_overall'] < 0.05).sum()}")
print(f"  Simpson's paradox flags: {df_sf['simpsons_flag'].sum()}")
print(f"  Mean |rho_overall|: {df_sf['rho_overall'].abs().mean():.3f}")
print(f"  Mean |rho_within|: {df_sf['rho_within_mean'].abs().mean():.3f}")
print()
print("Top 10 S-F correlations (by overall p-value):")
display_cols = [
    "structural",
    "functional",
    "rho_overall",
    "p_overall",
    "rho_within_mean",
    "n_within_significant",
    "simpsons_flag",
]
print(df_sf[display_cols].head(10).to_string(index=False))

S-F Edge: 80 metric pairs
  Significant overall (p<0.05): 59
  Simpson's paradox flags: 47
  Mean |rho_overall|: 0.516
  Mean |rho_within|: 0.267

Top 10 S-F correlations (by overall p-value):
             structural    functional  rho_overall    p_overall  rho_within_mean  n_within_significant  simpsons_flag
          edge_fraction size_fraction     1.000000 0.000000e+00         1.000000                     5          False
           active_heads       n_edges     0.983609 7.499348e-45         0.578194                     3          False
head_participation_rate size_fraction     0.978492 1.848061e-41         0.578194                     3          False
     universal_fraction size_fraction     0.962622 1.357000e-34         0.008937                     0           True
          skip_fraction       n_edges     0.961919 2.305581e-34         0.287308                     0           True
           active_heads   total_edges     0.955784 1.612490e-32              NaN                   

## 2. S-R Edge: Structure vs Representation

In [3]:
# Compute S-R edge
df_sr = compute_edge_correlations(
    df_pc, S_METRICS, R_METRICS, label_a="structural", label_b="representational"
)
df_sr = df_sr.sort_values("p_overall").reset_index(drop=True)
df_sr.to_csv(ANALYSIS_DIR / "triangle_sr_edge.csv", index=False)

print(f"S-R Edge: {len(df_sr)} metric pairs")
print(f"  Significant overall (p<0.05): {(df_sr['p_overall'] < 0.05).sum()}")
print(f"  Simpson's paradox flags: {df_sr['simpsons_flag'].sum()}")
print(f"  Mean |rho_overall|: {df_sr['rho_overall'].abs().mean():.3f}")
print(f"  Mean |rho_within|: {df_sr['rho_within_mean'].abs().mean():.3f}")
print()
print("Top 10 S-R correlations (by overall p-value):")
display_cols = [
    "structural",
    "representational",
    "rho_overall",
    "p_overall",
    "rho_within_mean",
    "n_within_significant",
    "simpsons_flag",
]
print(df_sr[display_cols].head(10).to_string(index=False))

S-R Edge: 80 metric pairs
  Significant overall (p<0.05): 69
  Simpson's paradox flags: 66
  Mean |rho_overall|: 0.601
  Mean |rho_within|: 0.160

Top 10 S-R correlations (by overall p-value):
             structural    representational  rho_overall    p_overall  rho_within_mean  n_within_significant  simpsons_flag
     universal_fraction    peak_probe_layer    -0.960981 4.610529e-34        -0.122008                     1           True
          edge_fraction    peak_probe_layer    -0.952837 1.005747e-31        -0.050381                     0           True
head_participation_rate    peak_probe_layer    -0.937175 3.305437e-28         0.184250                     0           True
     universal_fraction peak_probe_accuracy    -0.935714 6.308181e-28        -0.400000                     1           True
     universal_fraction       peak_mi_probe    -0.907143 1.806724e-23        -0.700000                     2          False
          edge_fraction peak_probe_accuracy    -0.906265 2.3439

## 3. R-F Edge: Representation vs Function

In [4]:
# Compute R-F edge
df_rf = compute_edge_correlations(
    df_pc, R_METRICS, F_METRICS, label_a="representational", label_b="functional"
)
df_rf = df_rf.sort_values("p_overall").reset_index(drop=True)
df_rf.to_csv(ANALYSIS_DIR / "triangle_rf_edge.csv", index=False)

print(f"R-F Edge: {len(df_rf)} metric pairs")
print(f"  Significant overall (p<0.05): {(df_rf['p_overall'] < 0.05).sum()}")
print(f"  Simpson's paradox flags: {df_rf['simpsons_flag'].sum()}")
print(f"  Mean |rho_overall|: {df_rf['rho_overall'].abs().mean():.3f}")
print(f"  Mean |rho_within|: {df_rf['rho_within_mean'].abs().mean():.3f}")
print()
print("Top 10 R-F correlations (by overall p-value):")
display_cols = [
    "representational",
    "functional",
    "rho_overall",
    "p_overall",
    "rho_within_mean",
    "n_within_significant",
    "simpsons_flag",
]
print(df_rf[display_cols].head(10).to_string(index=False))

R-F Edge: 64 metric pairs
  Significant overall (p<0.05): 51
  Simpson's paradox flags: 40
  Mean |rho_overall|: 0.526
  Mean |rho_within|: 0.144

Top 10 R-F correlations (by overall p-value):
   representational     functional  rho_overall    p_overall  rho_within_mean  n_within_significant  simpsons_flag
   peak_probe_layer  size_fraction    -0.952837 1.005747e-31        -0.050381                     0           True
peak_probe_accuracy  size_fraction    -0.906265 2.343950e-23         0.002962                     0           True
    peak_efficiency circuit_kl_div    -0.886762 4.267129e-21        -0.035479                     0           True
      peak_mi_probe  size_fraction    -0.877085 4.006332e-20        -0.038503                     0           True
   peak_probe_layer    total_edges     0.876147 4.927366e-20              NaN                     0          False
peak_mi_probe_layer  size_fraction    -0.873528 8.706907e-20         0.022565                     0           True
  

## 4. Predictive Power Comparison

For each model, use LOO-CV to compare how well the best structural vs best representational
metric predicts circuit quality (retention_ratio). Then compute incremental R²:
does adding one perspective improve prediction beyond the other?

In [5]:
def loo_cv_r2(X, y):
    """Compute LOO-CV R² for a linear regression."""
    if len(X.shape) == 1:
        X = X.reshape(-1, 1)
    loo = LeaveOneOut()
    y_pred = np.zeros_like(y, dtype=float)
    for train_idx, test_idx in loo.split(X):
        lr = LinearRegression()
        lr.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = lr.predict(X[test_idx])
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0


# Target: retention_ratio (primary functional quality metric)
TARGET = "retention_ratio"

# Compute predictor comparison per model and overall
pred_rows = []

for scope_label, df_scope in [("overall", df_pc)] + [
    (m, df_pc[df_pc["model"] == m]) for m in MODELS
]:
    if len(df_scope) < 4:
        continue

    y = df_scope[TARGET].values
    if np.std(y) < 1e-10:
        continue

    # Best single structural predictor
    best_s_metric, best_s_r2 = None, -np.inf
    for m in S_METRICS:
        x = df_scope[m].values
        if np.std(x) < 1e-10:
            continue
        r2 = loo_cv_r2(x, y)
        if r2 > best_s_r2:
            best_s_r2 = r2
            best_s_metric = m

    # Best single representational predictor
    best_r_metric, best_r_r2 = None, -np.inf
    for m in R_METRICS:
        x = df_scope[m].values
        if np.std(x) < 1e-10:
            continue
        r2 = loo_cv_r2(x, y)
        if r2 > best_r_r2:
            best_r_r2 = r2
            best_r_metric = m

    # All structural predictors
    s_cols = [m for m in S_METRICS if np.std(df_scope[m].values) > 1e-10]
    r_cols = [m for m in R_METRICS if np.std(df_scope[m].values) > 1e-10]

    # Incremental: S alone -> S+R
    r2_s_all = loo_cv_r2(df_scope[s_cols].values, y) if s_cols else np.nan
    r2_r_all = loo_cv_r2(df_scope[r_cols].values, y) if r_cols else np.nan
    sr_cols = s_cols + r_cols
    r2_sr_all = loo_cv_r2(df_scope[sr_cols].values, y) if sr_cols else np.nan

    pred_rows.append(
        {
            "scope": scope_label,
            "n": len(df_scope),
            "target": TARGET,
            "best_structural_metric": best_s_metric,
            "best_structural_r2": best_s_r2,
            "best_representational_metric": best_r_metric,
            "best_representational_r2": best_r_r2,
            "r2_all_structural": r2_s_all,
            "r2_all_representational": r2_r_all,
            "r2_combined": r2_sr_all,
            "delta_r2_adding_repr": r2_sr_all - r2_s_all
            if not np.isnan(r2_sr_all)
            else np.nan,
            "delta_r2_adding_struct": r2_sr_all - r2_r_all
            if not np.isnan(r2_sr_all)
            else np.nan,
        }
    )

df_pred = pd.DataFrame(pred_rows)
df_pred.to_csv(ANALYSIS_DIR / "incremental_prediction.csv", index=False)

print("Predictive Power Comparison (LOO-CV R²):")
print(
    df_pred[
        [
            "scope",
            "n",
            "best_structural_metric",
            "best_structural_r2",
            "best_representational_metric",
            "best_representational_r2",
        ]
    ].to_string(index=False)
)
print()
print("Incremental Prediction:")
print(
    df_pred[
        [
            "scope",
            "r2_all_structural",
            "r2_all_representational",
            "r2_combined",
            "delta_r2_adding_repr",
            "delta_r2_adding_struct",
        ]
    ].to_string(index=False)
)

Predictive Power Comparison (LOO-CV R²):
      scope  n best_structural_metric  best_structural_r2 best_representational_metric  best_representational_r2
    overall 60          skip_fraction            0.411610           final_prob_correct                  0.364393
pythia-1.4b 12          edge_fraction            0.452478            convergence_layer                  0.649620
pythia-160m 12          skip_fraction            0.283463            convergence_layer                  0.070418
  pythia-1b 12           mlp_fraction           -0.042274              peak_efficiency                 -0.121968
pythia-410m 12           mlp_fraction           -0.200209             peak_probe_layer                 -0.288468
 pythia-70m 12         resid_fraction           -0.092769        peak_separation_ratio                 -0.258249

Incremental Prediction:
      scope  r2_all_structural  r2_all_representational  r2_combined  delta_r2_adding_repr  delta_r2_adding_struct
    overall           0.7054

## 5. Visualizations

In [6]:
# --- VIZ T4_01: Similarity Triangle Diagram ---


def edge_summary(df_edge):
    """Compute summary stats for a triangle edge."""
    mean_rho_overall = df_edge["rho_overall"].abs().mean()
    mean_rho_within = df_edge["rho_within_mean"].abs().mean()
    n_sig_overall = (df_edge["p_overall"] < 0.05).sum()
    n_simpsons = df_edge["simpsons_flag"].sum()
    n_total = len(df_edge)
    # Mean within-model significance rate
    mean_within_sig_rate = (
        df_edge["n_within_significant"].mean() / df_edge["n_models"].mean()
        if df_edge["n_models"].mean() > 0
        else 0
    )
    return {
        "mean_rho_overall": mean_rho_overall,
        "mean_rho_within": mean_rho_within,
        "frac_sig_overall": n_sig_overall / n_total,
        "frac_simpsons": n_simpsons / n_total,
        "mean_within_sig_rate": mean_within_sig_rate,
    }


sf_sum = edge_summary(df_sf)
sr_sum = edge_summary(df_sr)
rf_sum = edge_summary(df_rf)

fig, ax = plt.subplots(1, 1, figsize=(8, 7))

# Triangle vertices
verts = {
    "S": np.array([0.0, 0.0]),
    "F": np.array([4.0, 0.0]),
    "R": np.array([2.0, 3.5]),
}

# Edge specs
edges = [
    ("S", "F", sf_sum, "S-F"),
    ("S", "R", sr_sum, "S-R"),
    ("R", "F", rf_sum, "R-F"),
]

for v1, v2, esum, label in edges:
    p1, p2 = verts[v1], verts[v2]

    # Color: green if within-model significant, orange if overall-only, red if neither
    if esum["mean_within_sig_rate"] > 0.5:
        color = "#2ca02c"  # green: robust within-model
    elif esum["frac_sig_overall"] > 0.5:
        color = "#ff7f0e"  # orange: overall-only (possibly confounded)
    else:
        color = "#d62728"  # red: not significant

    # Thickness proportional to mean overall |rho|
    lw = 2 + 8 * esum["mean_rho_overall"]

    ax.plot(
        [p1[0], p2[0]], [p1[1], p2[1]], "-", color=color, lw=lw, alpha=0.7, zorder=1
    )

    # Annotate at midpoint
    mid = (p1 + p2) / 2
    # Offset text perpendicular to edge
    direction = p2 - p1
    perp = np.array([-direction[1], direction[0]])
    perp = perp / np.linalg.norm(perp) * 0.35

    txt = f"{label}\n|r|={esum['mean_rho_overall']:.2f} (overall)\n|r|={esum['mean_rho_within']:.2f} (within)"
    if esum["frac_simpsons"] > 0.2:
        txt += f"\nSimpson's: {esum['frac_simpsons']:.0%}"

    ax.annotate(
        txt,
        xy=mid + perp,
        fontsize=8,
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.3", facecolor="white", edgecolor=color, alpha=0.9
        ),
    )

# Vertex labels
for name, pos in verts.items():
    full = {"S": "Structural", "F": "Functional", "R": "Representational"}[name]
    ax.plot(
        pos[0],
        pos[1],
        "o",
        color="white",
        markersize=35,
        markeredgecolor="black",
        markeredgewidth=2,
        zorder=5,
    )
    ax.text(
        pos[0],
        pos[1],
        f"{name}\n({full})",
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold",
        zorder=6,
    )

# Legend for edge colors
legend_patches = [
    mpatches.Patch(color="#2ca02c", label="Robust within-model"),
    mpatches.Patch(color="#ff7f0e", label="Overall only (possibly confounded)"),
    mpatches.Patch(color="#d62728", label="Not significant"),
]
ax.legend(handles=legend_patches, loc="lower center", fontsize=8, framealpha=0.9)

ax.set_xlim(-1, 5)
ax.set_ylim(-1, 4.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("S-F-R Similarity Triangle", fontsize=14, fontweight="bold", pad=10)

plt.savefig(VIZ_DIR / "T4_01_similarity_triangle.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: T4_01_similarity_triangle.png")

Saved: T4_01_similarity_triangle.png


In [7]:
# --- VIZ T4_02: Simpson's Paradox Scatter ---
# Show the strongest overall S-F correlation where Simpson's paradox occurs

simpsons_pairs = df_sf[df_sf["simpsons_flag"]].copy()
if len(simpsons_pairs) == 0:
    # Fall back to strongest overall S-F correlation
    simpsons_pairs = df_sf.head(1)

# Pick the one with largest |rho_overall|
top_pair = simpsons_pairs.loc[simpsons_pairs["rho_overall"].abs().idxmax()]
x_col = top_pair["structural"]
y_col = top_pair["functional"]

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Scatter by model
for model in MODELS:
    mask = df_pc["model"] == model
    ax.scatter(
        df_pc.loc[mask, x_col],
        df_pc.loc[mask, y_col],
        c=MODEL_COLORS[model],
        label=model,
        s=60,
        alpha=0.8,
        zorder=3,
    )

    # Within-model regression line
    x_m = df_pc.loc[mask, x_col].values
    y_m = df_pc.loc[mask, y_col].values
    if len(x_m) >= 3 and np.std(x_m) > 1e-10:
        slope, intercept, _, _, _ = stats.linregress(x_m, y_m)
        x_range = np.linspace(x_m.min(), x_m.max(), 50)
        ax.plot(
            x_range,
            slope * x_range + intercept,
            "--",
            color=MODEL_COLORS[model],
            alpha=0.5,
            lw=1.5,
        )

# Overall regression line
x_all = df_pc[x_col].values
y_all = df_pc[y_col].values
slope, intercept, _, _, _ = stats.linregress(x_all, y_all)
x_range = np.linspace(x_all.min(), x_all.max(), 50)
ax.plot(
    x_range,
    slope * x_range + intercept,
    "k-",
    alpha=0.7,
    lw=2,
    label=f"Overall (rho={top_pair['rho_overall']:.3f})",
)

ax.set_xlabel(x_col, fontsize=11)
ax.set_ylabel(y_col, fontsize=11)
is_simpsons = top_pair["simpsons_flag"]
title = (
    f"Simpson's Paradox: {x_col} vs {y_col}"
    if is_simpsons
    else f"Strongest S-F: {x_col} vs {y_col}"
)
ax.set_title(title, fontsize=12, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9)

# Annotate
rho_within = top_pair["rho_within_mean"]
ax.text(
    0.02,
    0.02,
    f"Overall rho = {top_pair['rho_overall']:.3f}\n"
    f"Mean within-model rho = {rho_within:.3f}\n"
    f"Simpson's paradox: {'YES' if is_simpsons else 'NO'}",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9),
)

plt.savefig(VIZ_DIR / "T4_02_simpsons_paradox.png", dpi=150, bbox_inches="tight")
plt.close()
print(
    f"Saved: T4_02_simpsons_paradox.png ({x_col} vs {y_col}, Simpson's={is_simpsons})"
)

Saved: T4_02_simpsons_paradox.png (universal_fraction vs size_fraction, Simpson's=True)


In [8]:
# --- VIZ T4_03: Edge Strength Comparison ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

edge_labels = ["S-F", "S-R", "R-F"]
edge_summaries = [sf_sum, sr_sum, rf_sum]

# Panel A: Overall vs Within-model |rho|
ax = axes[0]
x = np.arange(len(edge_labels))
width = 0.35
overall_vals = [s["mean_rho_overall"] for s in edge_summaries]
within_vals = [s["mean_rho_within"] for s in edge_summaries]

bars1 = ax.bar(
    x - width / 2, overall_vals, width, label="Overall", color="#1f77b4", alpha=0.8
)
bars2 = ax.bar(
    x + width / 2, within_vals, width, label="Within-model", color="#ff7f0e", alpha=0.8
)

ax.set_xlabel("Triangle Edge", fontsize=11)
ax.set_ylabel("Mean |Spearman rho|", fontsize=11)
ax.set_title("Edge Strength: Overall vs Within-Model", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(edge_labels)
ax.legend(fontsize=9)
ax.set_ylim(0, max(overall_vals + within_vals) * 1.3)

# Annotate bars
for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
for bar in bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Panel B: Simpson's paradox rate per edge
ax = axes[1]
simpsons_rates = [s["frac_simpsons"] for s in edge_summaries]
within_sig_rates = [s["mean_within_sig_rate"] for s in edge_summaries]

bars1 = ax.bar(
    x - width / 2,
    within_sig_rates,
    width,
    label="Within-model sig rate",
    color="#2ca02c",
    alpha=0.8,
)
bars2 = ax.bar(
    x + width / 2,
    simpsons_rates,
    width,
    label="Simpson's paradox rate",
    color="#d62728",
    alpha=0.8,
)

ax.set_xlabel("Triangle Edge", fontsize=11)
ax.set_ylabel("Rate", fontsize=11)
ax.set_title(
    "Robustness: Within-Model Significance & Simpson's Paradox",
    fontsize=12,
    fontweight="bold",
)
ax.set_xticks(x)
ax.set_xticklabels(edge_labels)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{bar.get_height():.0%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
for bar in bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{bar.get_height():.0%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig(
    VIZ_DIR / "T4_03_edge_strength_comparison.png", dpi=150, bbox_inches="tight"
)
plt.close()
print("Saved: T4_03_edge_strength_comparison.png")

Saved: T4_03_edge_strength_comparison.png


In [9]:
# --- VIZ T4_04: Predictor Comparison ---

fig, ax = plt.subplots(1, 1, figsize=(8, 5))

df_models_pred = df_pred[df_pred["scope"] != "overall"].copy()
x = np.arange(len(df_models_pred))
width = 0.35

bars1 = ax.bar(
    x - width / 2,
    df_models_pred["best_structural_r2"].values,
    width,
    label="Best structural",
    color="#1f77b4",
    alpha=0.8,
)
bars2 = ax.bar(
    x + width / 2,
    df_models_pred["best_representational_r2"].values,
    width,
    label="Best representational",
    color="#2ca02c",
    alpha=0.8,
)

ax.set_xlabel("Model", fontsize=11)
ax.set_ylabel(f"LOO-CV R² (predicting {TARGET})", fontsize=11)
ax.set_title(
    "Best Single Predictor: Structural vs Representational",
    fontsize=12,
    fontweight="bold",
)
ax.set_xticks(x)
ax.set_xticklabels(df_models_pred["scope"].values, fontsize=10)
ax.legend(fontsize=9)
ax.axhline(y=0, color="gray", linestyle="-", alpha=0.3)

# Annotate with metric names
for i, (_, row) in enumerate(df_models_pred.iterrows()):
    s_name = str(row["best_structural_metric"]).replace("_", "\n")
    r_name = str(row["best_representational_metric"]).replace("_", "\n")
    ax.text(
        i - width / 2,
        max(0, row["best_structural_r2"]) + 0.02,
        f"{row['best_structural_r2']:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
    ax.text(
        i + width / 2,
        max(0, row["best_representational_r2"]) + 0.02,
        f"{row['best_representational_r2']:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.savefig(VIZ_DIR / "T4_04_predictor_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: T4_04_predictor_comparison.png")

Saved: T4_04_predictor_comparison.png


## 6. Summary

In [10]:
print("=" * 80)
print("PHASE 4: INTEGRATION 2: SIMILARITY TRIANGLE")
print("=" * 80)

print(f"\n--- Triangle Edge Summary ---")
for label, esum, df_edge in [
    ("S-F", sf_sum, df_sf),
    ("S-R", sr_sum, df_sr),
    ("R-F", rf_sum, df_rf),
]:
    print(f"\n  {label} Edge ({len(df_edge)} metric pairs):")
    print(f"    Mean |rho| overall:      {esum['mean_rho_overall']:.3f}")
    print(f"    Mean |rho| within-model: {esum['mean_rho_within']:.3f}")
    print(f"    Significant overall:     {esum['frac_sig_overall']:.0%}")
    print(f"    Within-model sig rate:   {esum['mean_within_sig_rate']:.0%}")
    print(f"    Simpson's paradox rate:  {esum['frac_simpsons']:.0%}")

print(f"\n--- Predictor Comparison ---")
overall_row = df_pred[df_pred["scope"] == "overall"].iloc[0]
print(f"  Overall (N={int(overall_row['n'])}):")
print(
    f"    Best structural:       {overall_row['best_structural_metric']} (R²={overall_row['best_structural_r2']:.3f})"
)
print(
    f"    Best representational: {overall_row['best_representational_metric']} (R²={overall_row['best_representational_r2']:.3f})"
)
print(f"    All structural:        R²={overall_row['r2_all_structural']:.3f}")
print(f"    All representational:  R²={overall_row['r2_all_representational']:.3f}")
print(f"    Combined:              R²={overall_row['r2_combined']:.3f}")
print(f"    ΔR² adding repr:       {overall_row['delta_r2_adding_repr']:+.3f}")
print(f"    ΔR² adding struct:     {overall_row['delta_r2_adding_struct']:+.3f}")

print(f"\n--- Output Files ---")
for f in sorted(ANALYSIS_DIR.glob("triangle_*")) + sorted(
    ANALYSIS_DIR.glob("incremental_*")
):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")
for f in sorted(VIZ_DIR.glob("T4_0*")):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

print("\nDone.")

PHASE 4: INTEGRATION 2: SIMILARITY TRIANGLE

--- Triangle Edge Summary ---

  S-F Edge (80 metric pairs):
    Mean |rho| overall:      0.516
    Mean |rho| within-model: 0.267
    Significant overall:     74%
    Within-model sig rate:   26%
    Simpson's paradox rate:  59%

  S-R Edge (80 metric pairs):
    Mean |rho| overall:      0.601
    Mean |rho| within-model: 0.160
    Significant overall:     86%
    Within-model sig rate:   16%
    Simpson's paradox rate:  82%

  R-F Edge (64 metric pairs):
    Mean |rho| overall:      0.526
    Mean |rho| within-model: 0.144
    Significant overall:     80%
    Within-model sig rate:   13%
    Simpson's paradox rate:  62%

--- Predictor Comparison ---
  Overall (N=60):
    Best structural:       skip_fraction (R²=0.412)
    Best representational: final_prob_correct (R²=0.364)
    All structural:        R²=0.705
    All representational:  R²=0.535
    Combined:              R²=0.663
    ΔR² adding repr:       -0.043
    ΔR² adding struct:    